<a href="https://colab.research.google.com/github/ulka29/paper/blob/main/06_Linguistic_Profanity%26Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
path = "https://raw.githubusercontent.com/ulka29/paper/7aad80ab3fe4ad5ffeabffb7c30bcb70edb218bd/01_data/processed/df_cleaned.csv"# import Check dataset
df= pd.read_csv(path)
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258


In [ ]:
# Select relevant features for SA + ML
df_ml = df[
    [
        'lemmatized_tweet',
        'negative',
        'positive',
        'neutral',
        'compound',
        'cyberbullying_type'
    ]
].copy()

# Check shape
print("\n:-------------Shape::-------------", df_ml.shape)

# Check column names
print("\n:-------------Columns::-------------")
print(df_ml.columns.tolist())

# Check missing values
print("\n----------------Missing values:-------------")
print(df_ml.isnull().sum())

# Display first 5 rows
df_ml.head()


:-------------Shape::------------- (42645, 6)

:-------------Columns::-------------
['lemmatized_tweet', 'negative', 'positive', 'neutral', 'compound', 'cyberbullying_type']

----------------Missing values:-------------
lemmatized_tweet      30
negative               0
positive               0
neutral                0
compound               0
cyberbullying_type     0
dtype: int64


,lemmatized_tweet,negative,positive,neutral,compound,cyberbullying_type
0,word katandandre food crapilicious mkr,0.000,0.000,1.000,0.0000,not_cyberbullying
1,aussietv white mkr theblock imacelebrityau tod...,0.000,0.000,1.000,0.0000,not_cyberbullying
2,classy whore red velvet cupcake,0.320,0.199,0.481,-0.4137,not_cyberbullying
3,meh p thanks head concerned another angry dude...,0.243,0.150,0.607,-0.4854,not_cyberbullying
4,isi account pretending kurdish account like is...,0.129,0.180,0.691,0.0258,not_cyberbullying


In [ ]:
PROFANITY_WORDS = {
    'fuck', 'fucking', 'fucked', 'fuckin',
    'shit', 'shitty',
    'bitch', 'bitches',
    'asshole', 'assholes',
    'bastard',
    'damn',
    'dick',
    'piss',
    'crap',
    'bullshit',
    'motherfucker',
    'idiot',
    'stupid',
    'moron',
    'retard',
    'retarded'
}

In [ ]:
def extract_linguistic_features(text):

    text = str(text)

    # Basic tokens
    words = re.findall(r'\b\w+\b', text.lower())

    word_count = len(words)
    char_count = len(text)
    unique_words = len(set(words))

    # Profanity
    profanity_words = [
        word for word in words
        if word in PROFANITY_WORDS
    ]

    profanity_count = len(profanity_words)
    unique_profanity_count = len(set(profanity_words))

    profanity_ratio = (
        profanity_count / word_count
        if word_count > 0 else 0
    )

    profanity_presence = int(profanity_count > 0)

    # Second-person language
    second_person_words = {
        'you',
        'your',
        'yours',
        'yourself',
        'yourselves',
        'youre',
        'youve',
        'youll',
        'youd'
    }

    second_person_count = sum(
        word in second_person_words
        for word in words
    )

    # Punctuation
    exclamation_count = text.count('!')
    question_count = text.count('?')
    comma_count = text.count(',')
    period_count = text.count('.')

    repeated_exclamation_count = len(
        re.findall(r'!{2,}', text)
    )

    repeated_question_count = len(
        re.findall(r'\?{2,}', text)
    )

    repeated_punctuation_count = len(
        re.findall(r'[!?]{2,}', text)
    )

    # Mentions / Hashtags
    mention_count = len(
        re.findall(r'@\w+', text)
    )

    hashtag_count = len(
        re.findall(r'#\w+', text)
    )

    # Return features
    return pd.Series({
        'word_count': word_count,
        'char_count': char_count,
        'unique_word_count': unique_words,

        'profanity_count': profanity_count,
        'unique_profanity_count': unique_profanity_count,
        'profanity_ratio': profanity_ratio,
        'profanity_presence': profanity_presence,

        'second_person_count': second_person_count,

        'exclamation_count': exclamation_count,
        'question_count': question_count,
        'comma_count': comma_count,
        'period_count': period_count,

        'repeated_exclamation_count': repeated_exclamation_count,
        'repeated_question_count': repeated_question_count,
        'repeated_punctuation_count': repeated_punctuation_count,

        'mention_count': mention_count,
        'hashtag_count': hashtag_count
    })


In [ ]:
import re

In [ ]:
linguistic_features = df['tweet_text'].apply(
    extract_linguistic_features
)

df = pd.concat(
    [df, linguistic_features],
    axis=1
)

In [ ]:
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound,word_count,...,second_person_count,exclamation_count,question_count,comma_count,period_count,repeated_exclamation_count,repeated_question_count,repeated_punctuation_count,mention_count,hashtag_count
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,9.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,14.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137,9.0,...,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0


In [ ]:
print("Class-wise counts:")
for class_name, count in df['cyberbullying_type'].value_counts().items():
    print(f"{class_name}: {count}")

Class-wise counts:
religion: 7964
age: 7957
gender: 7491
ethnicity: 7427
not_cyberbullying: 6445
other_cyberbullying: 5361


In [ ]:
print(df.columns.tolist())

['tweet_text', 'cyberbullying_type', 'clean_tweet', 'lemmatized_tweet', 'sentiment_scores', 'negative', 'positive', 'neutral', 'compound', 'word_count', 'char_count', 'unique_word_count', 'profanity_count', 'unique_profanity_count', 'profanity_ratio', 'profanity_presence', 'second_person_count', 'exclamation_count', 'question_count', 'comma_count', 'period_count', 'repeated_exclamation_count', 'repeated_question_count', 'repeated_punctuation_count', 'mention_count', 'hashtag_count']


In [ ]:

# ============================================================
# WORD TF-IDF + SENTIMENT + BEHAVIORAL FEATURES
# Models: Logistic Regression and Linear SVM
# TF-IDF: 10,000 features, n-gram range (1, 2)
# Class imbalance: Balanced class weights
# Maximum iterations: 10,000
# ============================================================

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.base import clone

# ------------------------------------------------------------
# 1. SELECT FEATURES
# ------------------------------------------------------------

text_col = "lemmatized_tweet"

numeric_features = [
    "negative",
    "positive",
    "neutral",
    "compound",
    "second_person_count",
    "mention_count",
    "hashtag_count",
    "profanity_count",
    "unique_profanity_count",
    "profanity_ratio",
    "profanity_presence"
]

target_col = "cyberbullying_type"

df_exp = df[
    [text_col] + numeric_features + [target_col]
].copy()

# Remove rows without target labels
df_exp = df_exp.dropna(subset=[target_col])

# Text preprocessing
X_text = df_exp[text_col].fillna("").astype(str)

# Convert numeric features safely
X_numeric = (
    df_exp[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(
    df_exp[target_col].astype(str)
)

print("Classes:", list(label_encoder.classes_))
print("Dataset shape:", df_exp.shape)

# ------------------------------------------------------------
# 2. STRATIFIED TRAIN-TEST SPLIT
# ------------------------------------------------------------

(
    X_text_train,
    X_text_test,
    X_num_train,
    X_num_test,
    y_train,
    y_test
) = train_test_split(
    X_text,
    X_numeric,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Sparse numeric feature matrices
N_train = csr_matrix(
    X_num_train.to_numpy(dtype=float)
)
N_test = csr_matrix(
    X_num_test.to_numpy(dtype=float)
)

print("\nTraining samples:", len(y_train))
print("Testing samples :", len(y_test))

# ------------------------------------------------------------
# 3. WORD TF-IDF
# Fit ONLY on training text
# ------------------------------------------------------------

tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True
)

W_train = tfidf.fit_transform(X_text_train)
W_test = tfidf.transform(X_text_test)

print("\nWord TF-IDF features:", W_train.shape[1])

# ------------------------------------------------------------
# 4. COMBINE TEXT + NUMERIC FEATURES
# ------------------------------------------------------------

X_train = hstack(
    [W_train, N_train],
    format="csr"
)

X_test = hstack(
    [W_test, N_test],
    format="csr"
)

print("Combined training shape:", X_train.shape)
print("Combined testing shape :", X_test.shape)

# ------------------------------------------------------------
# 5. MODELS
# ------------------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=10000,
        C=1.0,
        solver="lbfgs",
        random_state=42
    ),

    "Linear SVM": LinearSVC(
        class_weight="balanced",
        max_iter=10000,
        C=0.1,
        random_state=42
    )
}

# ------------------------------------------------------------
# 6. TRAIN + EVALUATE
# ------------------------------------------------------------

results = []
reports = {}
predictions = {}
confusion_matrices = {}
fitted_models = {}

for model_name, model_template in models.items():

    print("\n" + "=" * 65)
    print(model_name)
    print("=" * 65)

    model = clone(model_template)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    fitted_models[model_name] = model
    predictions[model_name] = y_pred

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(
        y_test, y_pred,
        average="macro",
        zero_division=0
    )
    weighted_f1 = f1_score(
        y_test, y_pred,
        average="weighted",
        zero_division=0
    )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Macro-F1": macro_f1,
        "Weighted-F1": weighted_f1
    })

    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Macro-F1    : {macro_f1:.4f}")
    print(f"Weighted-F1 : {weighted_f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        zero_division=0
    ))

    reports[model_name] = classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0
    )

    confusion_matrices[model_name] = confusion_matrix(
        y_test,
        y_pred,
        labels=np.arange(len(label_encoder.classes_))
    )

# ------------------------------------------------------------
# 7. COMPARISON TABLE
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        "Macro-F1",
        ascending=False
    ).reset_index(drop=True).round(4)
)

# ------------------------------------------------------------
# 8. NOT-CYBERBULLYING ANALYSIS
# ------------------------------------------------------------

focus_class = "not_cyberbullying"

if focus_class in label_encoder.classes_:

    print("\nNOT-CYBERBULLYING CLASS PERFORMANCE")
    print("-" * 65)

    for model_name, report in reports.items():

        metrics = report[focus_class]

        print(
            f"{model_name}: "
            f"Precision={metrics['precision']:.4f}, "
            f"Recall={metrics['recall']:.4f}, "
            f"F1={metrics['f1-score']:.4f}"
        )

# ------------------------------------------------------------
# 9. CONFUSION MATRICES
# ------------------------------------------------------------

for model_name, cm in confusion_matrices.items():

    print(f"\nConfusion matrix: {model_name}")
    print(pd.DataFrame(
        cm,
        index=label_encoder.classes_,
        columns=label_encoder.classes_
    ))


Classes: ['age', 'ethnicity', 'gender', 'not_cyberbullying', 'other_cyberbullying', 'religion']
Dataset shape: (42645, 13)

Training samples: 29851
Testing samples : 12794

Word TF-IDF features: 10000
Combined training shape: (29851, 10011)
Combined testing shape : (12794, 10011)

Logistic Regression
Accuracy    : 0.8499
Macro-F1    : 0.8332
Weighted-F1 : 0.8524

Classification Report:
                     precision    recall  f1-score   support

                age       0.96      0.97      0.96      2387
          ethnicity       0.98      0.98      0.98      2228
             gender       0.93      0.84      0.88      2248
  not_cyberbullying       0.60      0.61      0.61      1934
other_cyberbullying       0.57      0.65      0.61      1608
           religion       0.96      0.95      0.95      2389

           accuracy                           0.85     12794
          macro avg       0.84      0.83      0.83     12794
       weighted avg       0.86      0.85      0.85     12794

,Model,Accuracy,Macro-F1,Weighted-F1
0,Linear SVM,0.8524,0.8334,0.8527
1,Logistic Regression,0.8499,0.8332,0.8524



NOT-CYBERBULLYING CLASS PERFORMANCE
-----------------------------------------------------------------
Logistic Regression: Precision=0.6041, Recall=0.6075, F1=0.6058
Linear SVM: Precision=0.6196, Recall=0.5879, F1=0.6033

Confusion matrix: Logistic Regression
                      age  ethnicity  gender  not_cyberbullying  \
age                  2313          3       5                 40   
ethnicity               3       2184       3                 11   
gender                  8          7    1888                185   
not_cyberbullying      60         18      67               1175   
other_cyberbullying    23         17      57                448   
religion                1          2       6                 86   

                     other_cyberbullying  religion  
age                                   26         0  
ethnicity                             24         3  
gender                               153         7  
not_cyberbullying                    546        68  
othe

In [ ]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [ ]:

# ============================================================
# EXPERIMENT 2: ADDITIONAL MODELS + GRID SEARCH CV
# Existing split: X_train, X_test, y_train, y_test
# Selection metric: Macro-F1
# CV: 3-fold stratified cross-validation
# ============================================================

import time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.base import clone
from sklearn.naive_bayes import MultinomialNB
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_sample_weight
from tqdm.auto import tqdm
from IPython.display import display





# ------------------------------------------------------------
# 1. MODEL-SPECIFIC INPUT MATRICES
# ------------------------------------------------------------
# All models now use the full combined feature matrix:
# word TF-IDF + sentiment + behavioral features.

model_inputs = {
    "1. XGBoost": (X_train, X_test),
    "2. SGD Classifier": (X_train, X_test),
    "3. Random Forest": (X_train, X_test),
    "4. Extra Trees": (X_train, X_test),
    "5. MLP Neural Network": (X_train, X_test)
}

# ------------------------------------------------------------
# 2. DEFINE ESTIMATORS AND SEARCH SPACES
# ------------------------------------------------------------

model_configs = {
    "1. XGBoost": {
        "estimator": XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=42,
            n_jobs=1
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [3, 6],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8],
            "colsample_bytree": [0.8]
        }
    },

    "2. SGD Classifier": {
        "estimator": SGDClassifier(
            class_weight="balanced",
            max_iter=10000,
            tol=1e-3,
            random_state=42
        ),
        "params": {
            "loss": ["hinge", "log_loss"],
            "alpha": [1e-4, 1e-3, 1e-2],
            "penalty": ["l2"]
        }
    },

    "3. Random Forest": {
        "estimator": RandomForestClassifier(
            class_weight="balanced",
            n_jobs=1,
            random_state=42
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [20, 40],
            "min_samples_leaf": [1, 2]
        }
    },

    "4. Extra Trees": {
        "estimator": ExtraTreesClassifier(
            class_weight="balanced",
            n_jobs=1,
            random_state=42
        ),
        "params": {
            "n_estimators": [100, 200],
            "max_depth": [20, 40],
            "min_samples_leaf": [1, 2]
        }
    },

    "5. MLP Neural Network": {
        "estimator": Pipeline([
            ("scaler", MaxAbsScaler()),
            ("mlp", MLPClassifier(
                solver="adam",
                activation="relu",
                batch_size=128,
                max_iter=100,
                early_stopping=True,
                n_iter_no_change=8,
                random_state=42
            ))
        ]),
        "params": {
            "mlp__hidden_layer_sizes": [(64,), (128,)],
            "mlp__alpha": [1e-4, 1e-3]
        }
    }
}


# ------------------------------------------------------------
# 3. STRATIFIED CROSS-VALIDATION
# ------------------------------------------------------------

cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

# ------------------------------------------------------------
# 4. GRID SEARCH + FINAL TEST EVALUATION
# ------------------------------------------------------------

tuned_results = []
tuned_models = {}
tuned_predictions = {}
tuned_reports = {}
tuned_confusion_matrices = {}
grid_search_objects = {}


progress_bar = tqdm(
    model_configs.items(),
    total=len(model_configs),
    desc="Overall Grid Search",
    unit="model"
)

for model_name, config in progress_bar:

    progress_bar.set_postfix_str(f"Running: {model_name}")

    print("\n" + "=" * 75)
    print("MODEL:", model_name)
    print("=" * 75)

    Xtr, Xte = model_inputs[model_name]
    estimator = clone(config["estimator"])

    search = GridSearchCV(
        estimator=estimator,
        param_grid=config["params"],
        scoring="f1_macro",
        cv=cv_strategy,
        refit=True,
        n_jobs=1,
        verbose=1,
        error_score="raise",
        return_train_score=False
    )

    start_time = time.time()

    try:
        # Balance NB training samples through sample weights.
        # Other supported models use class_weight internally.
        if model_name == "1. Multinomial NB":
            sample_weights = compute_sample_weight(
                class_weight="balanced",
                y=y_train
            )
            search.fit(
                Xtr,
                y_train,
                sample_weight=sample_weights
            )
        else:
            search.fit(Xtr, y_train)

        elapsed = time.time() - start_time

        # GridSearchCV refits the best estimator on all training data
        best_model = search.best_estimator_
        y_pred = best_model.predict(Xte)

        accuracy = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(
            y_test, y_pred,
            average="macro",
            zero_division=0
        )
        weighted_f1 = f1_score(
            y_test, y_pred,
            average="weighted",
            zero_division=0
        )

        tuned_results.append({
            "Model": model_name,
            "Best CV Macro-F1": search.best_score_,
            "Test Accuracy": accuracy,
            "Test Macro-F1": macro_f1,
            "Test Weighted-F1": weighted_f1,
            "Time (seconds)": round(elapsed, 2),
            "Status": "Completed"
        })

        tuned_models[model_name] = best_model
        tuned_predictions[model_name] = y_pred
        grid_search_objects[model_name] = search

        tuned_reports[model_name] = classification_report(
            y_test,
            y_pred,
            labels=np.arange(len(label_encoder.classes_)),
            target_names=label_encoder.classes_,
            output_dict=True,
            zero_division=0
        )

        tuned_confusion_matrices[model_name] = confusion_matrix(
            y_test,
            y_pred,
            labels=np.arange(len(label_encoder.classes_))
        )

        print("\nBest parameters:")
        print(search.best_params_)

        print(f"\nBest CV Macro-F1: {search.best_score_:.4f}")
        print(f"Test Accuracy   : {accuracy:.4f}")
        print(f"Test Macro-F1   : {macro_f1:.4f}")
        print(f"Test Weighted-F1: {weighted_f1:.4f}")
        print(f"Elapsed time    : {elapsed:.2f} seconds")

        print("\nTest classification report:")
        print(classification_report(
            y_test,
            y_pred,
            labels=np.arange(len(label_encoder.classes_)),
            target_names=label_encoder.classes_,
            zero_division=0
        ))

    except Exception as e:
        print("Model failed:", repr(e))

        tuned_results.append({
            "Model": model_name,
            "Best CV Macro-F1": np.nan,
            "Test Accuracy": np.nan,
            "Test Macro-F1": np.nan,
            "Test Weighted-F1": np.nan,
            "Time (seconds)": round(time.time() - start_time, 2),
            "Status": f"Failed: {str(e)[:150]}"
        })

# ------------------------------------------------------------
# 5. COMPARE RESULTS
# ------------------------------------------------------------

tuned_results_df = pd.DataFrame(tuned_results)

display(
    tuned_results_df.sort_values(
        "Test Macro-F1",
        ascending=False,
        na_position="last"
    ).reset_index(drop=True).round(4)
)

# ------------------------------------------------------------
# 6. NOT-CYBERBULLYING CLASS PERFORMANCE
# ------------------------------------------------------------

focus_class = "not_cyberbullying"

if focus_class in label_encoder.classes_:

    print("\nNOT-CYBERBULLYING PERFORMANCE")
    print("-" * 85)

    for model_name, report in tuned_reports.items():
        m = report[focus_class]

        print(
            f"{model_name:27s} | "
            f"Precision: {m['precision']:.4f} | "
            f"Recall: {m['recall']:.4f} | "
            f"F1: {m['f1-score']:.4f}"
        )

# ------------------------------------------------------------
# 7. CONFUSION MATRICES
# ------------------------------------------------------------

for model_name, cm in tuned_confusion_matrices.items():

    print("\n", model_name)

    display(pd.DataFrame(
        cm,
        index=label_encoder.classes_,
        columns=label_encoder.classes_
    ))



MODEL: 1. XGBoost
Fitting 3 folds for each of 8 candidates, totalling 24 fits
